Cell 1 — Import + check FAISS

In [ ]:
import os
import time
import numpy as np
import pandas as pd

from typing import Dict, Tuple, List

try:
    import faiss
    print(" FAISS version:", faiss.__version__)
except Exception as e:
    print("FAISS not available. Error:", e)
    print("Install with: pip install faiss-cpu")

 FAISS version: 1.13.2


Cell 2 — Config 

In [ ]:
emb_folder = "\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}

TOPK_LIST = [1, 5, 10, 20]

OUT_XLSX = "faiss_flatl2_results.xlsx"

Cell 3 — Loader (embedding + label + id)

In [5]:
def is_numeric_col_name(c) -> bool:
    if isinstance(c, (int, np.integer)):
        return True
    s = str(c)
    return s.isdigit()

def load_embedding_xlsx(path: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label", "y", "class")]
    if not label_candidates:
        raise ValueError(f"[{path}] No label column found.")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename", "file", "text_file", "id", "file_id")]
    if id_candidates:
        preferred = [c for c in id_candidates if str(c).lower() in ("filename", "file", "text_file")]
        id_col = preferred[0] if preferred else id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col and not is_numeric_col_name(c)]
        if not non_num:
            raise ValueError(f"[{path}] No id-like column found.")
        id_col = non_num[0]

    # embedding columns: numeric OR e0..eN
    emb_cols = [c for c in df.columns if c != label_col and is_numeric_col_name(c)]
    if not emb_cols:
        emb_cols = [c for c in df.columns
                    if c != label_col and str(c).lower().startswith("e") and str(c)[1:].isdigit()]

    if not emb_cols:
        # کمک برای دیباگ: چند ستون اول را نشان بده
        raise ValueError(f"[{path}] No embedding columns found. "
                         f"Columns seen (first 20): {df.columns.tolist()[:20]}")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df[label_col].to_numpy()
    ids = df[id_col].astype(str).to_numpy()

    return df, X, y, ids

Cell 4 — FAISS IndexFlatL2 

In [ ]:
import faiss

emb_name = "sbert"
path = EMBEDDING_FILES[emb_name]

df0, X0, y0, ids0 = load_embedding_xlsx(path)

# dims
n, d = X0.shape
print("Loaded:", emb_name, "| N:", n, "| d:", d)

index = faiss.IndexFlatL2(d)
index.add(X0)  # add all vectors
print("FAISS index ntotal:", index.ntotal)

#  test a query
k_test = 6
D, I = index.search(X0[0:1], k_test)
print("Neighbors indices:", I[0])
print("Distances (L2^2):", D[0])
print("Query id:", ids0[0])
print("Top neighbor ids:", [ids0[j] for j in I[0]])

Loaded: sbert | N: 2323 | d: 384
FAISS index ntotal: 2323
Neighbors indices: [   0    6 1081  765 2109  468]
Distances (L2^2): [0.         0.00143163 0.02108789 0.0249812  0.0255491  0.02707508]
Query id: 1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt
Top neighbor ids: ['1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt', '1009_78_shopping_e65305c2-a92b-4c68-91cf-b2cbf57a3c77.txt', '2582_78_shopping_94cd5902-308a-435a-91ec-5b89ac453677.txt', '2236_78_shopping__ufhgYO3vEemVV_B95cpSSw.txt', '499_78_shopping_f7cb9ede-73f5-4d19-babe-f5969f97e197.txt', '1715_78_shopping_5f377d1f-ad57-4e86-8585-12b77f0896aa.txt']


Cell 5 — IR Metrics

In [7]:
import numpy as np

def precision_at_k(rels: np.ndarray, k: int) -> float:
    return float(np.sum(rels[:k])) / float(k)

def recall_at_k(rels: np.ndarray, k: int, total_relevant: int) -> float:
    if total_relevant <= 0:
        return 0.0
    return float(np.sum(rels[:k])) / float(total_relevant)

def dcg_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    denom = np.log2(np.arange(2, k + 2))
    return float(np.sum(rels_k / denom))

def ndcg_at_k(rels: np.ndarray, k: int) -> float:
    dcg = dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else (dcg / idcg)

def mrr_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    idx = np.where(rels_k == 1)[0]
    return 0.0 if len(idx) == 0 else (1.0 / float(idx[0] + 1))

Cell 6 — FAISS FlatL2 Evaluation

In [11]:
import faiss
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from typing import List

def faiss_flat_eval_all3(
    X: np.ndarray, y: np.ndarray, ids: np.ndarray,
    topk_list: List[int]
) -> pd.DataFrame:
    """
    Exact search with FAISS:
      - IndexFlatL2        -> metric='L2'
      - IndexFlatIP        -> metric='dot'
      - IndexFlatIP + L2-normalize -> metric='cosine'
    Relevance: same label.
    Returns PerQuery rows compatible with previous experiments.
    """
    X = np.asarray(X, dtype=np.float32)
    N, d = X.shape

    maxK = max(topk_list)
    search_k = min(N, maxK + 1)  # +1 to remove self

    rows = []

    def eval_with_index(X_use: np.ndarray, index, metric_name: str):
        index.reset()
        index.add(X_use)

        # batch search
        D, I = index.search(X_use, search_k)

        for i in range(N):
            nbrs = I[i]
            nbrs = nbrs[nbrs != i]   # remove self
            if len(nbrs) == 0:
                continue
            nbrs = nbrs[:maxK]

            rel = (y[nbrs] == y[i]).astype(np.int32)
            total_rel = int(np.sum(y == y[i]) - 1)

            for K in topk_list:
                Ke = min(K, len(rel))
                rows.append({
                    "method": "faiss_flat",
                    "metric": metric_name,
                    "query_id": ids[i],
                    "query_label": int(y[i]),
                    "K": int(K),
                    "precision@K": precision_at_k(rel, Ke),
                    "recall@K": recall_at_k(rel, Ke, total_rel),
                    "ndcg@K": ndcg_at_k(rel, Ke),
                    "mrr@K": mrr_at_k(rel, Ke),
                })

    # 1) L2
    index_l2 = faiss.IndexFlatL2(d)
    eval_with_index(X, index_l2, "L2")

    # 2) Dot product (Inner Product)
    index_ip = faiss.IndexFlatIP(d)
    eval_with_index(X, index_ip, "dot")

    # 3) Cosine = Inner Product on normalized vectors
    Xn = normalize(X, axis=1).astype(np.float32)
    index_cos = faiss.IndexFlatIP(d)
    eval_with_index(Xn, index_cos, "cosine")

    return pd.DataFrame(rows)

Cell 7 — تست روی SBERT (Sanity check)

In [13]:
emb_name = "sbert"
path = EMBEDDING_FILES[emb_name]
df0, X0, y0, ids0 = load_embedding_xlsx(path)

t0 = time.time()
perq_sbert = faiss_flat_eval_all3(X0, y0, ids0, TOPK_LIST)
print("Done:", emb_name, "| rows:", len(perq_sbert), "| seconds:", round(time.time()-t0, 2))

perq_sbert.head()

Done: sbert | rows: 27876 | seconds: 0.9


,method,metric,query_id,query_label,K,precision@K,recall@K,ndcg@K,mrr@K
0,faiss_flat,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,1,1.0,0.007463,1.0,1.0
1,faiss_flat,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,5,1.0,0.037313,1.0,1.0
2,faiss_flat,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,10,1.0,0.074627,1.0,1.0
3,faiss_flat,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,20,1.0,0.149254,1.0,1.0
4,faiss_flat,L2,1001_18_computer__wJS0wCqtEeiGfabwoJ7AXg.txt,18,1,1.0,0.006452,1.0,1.0


In [14]:
perq_sbert["metric"].value_counts()

metric
L2        9292
dot       9292
cosine    9292
Name: count, dtype: int64

Cell 8 —   embeddingها + Summary + save Excel

In [ ]:
def summarize(perquery: pd.DataFrame, embedding_name: str) -> pd.DataFrame:
    grp = perquery.groupby(["method", "metric", "K"], as_index=False).agg({
        "precision@K": "mean",
        "recall@K": "mean",
        "ndcg@K": "mean",
        "mrr@K": "mean",
    })
    grp.insert(0, "embedding", embedding_name)
    return grp

all_perquery = []
all_summary = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"\n=== FAISS IndexFlatL2: {emb_name} ===")
    if not os.path.exists(path):
        print("❌ File not found:", path)
        continue

    df_e, X, y, ids = load_embedding_xlsx(path)

    t0 = time.time()
    perq = faiss_flat_eval_all3(X, y, ids, TOPK_LIST)
    perq.insert(0, "embedding", emb_name)
    all_perquery.append(perq)

    summ = summarize(perq, emb_name)
    all_summary.append(summ)

    print("perquery rows:", len(perq), "| seconds:", round(time.time()-t0, 2))

df_perquery = pd.concat(all_perquery, ignore_index=True)
df_summary  = pd.concat(all_summary, ignore_index=True)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    df_summary.to_excel(w, sheet_name="Summary", index=False)
    df_perquery.to_excel(w, sheet_name="PerQuery", index=False)

print("\n Saved:", OUT_XLSX)
df_summary.head(20)


=== FAISS IndexFlatL2: bert_finetuned ===
   perquery rows: 27876 | seconds: 2.34

=== FAISS IndexFlatL2: gemini ===
   perquery rows: 27876 | seconds: 1.36

=== FAISS IndexFlatL2: qwen3_8b ===
   perquery rows: 27876 | seconds: 1.84

=== FAISS IndexFlatL2: sbert ===
   perquery rows: 27876 | seconds: 0.84

 Saved: faiss_flatl2_results.xlsx


,embedding,method,metric,K,precision@K,recall@K,ndcg@K,mrr@K
0,bert_finetuned,faiss_flat,L2,1,0.950495,0.013989,0.950495,0.950495
1,bert_finetuned,faiss_flat,L2,5,0.934998,0.068434,0.944677,0.960877
2,bert_finetuned,faiss_flat,L2,10,0.930822,0.135849,0.946978,0.961511
3,bert_finetuned,faiss_flat,L2,20,0.925398,0.269264,0.964025,0.961814
4,bert_finetuned,faiss_flat,cosine,1,0.954369,0.014066,0.954369,0.954369
5,bert_finetuned,faiss_flat,cosine,5,0.935601,0.068489,0.945804,0.962692
6,bert_finetuned,faiss_flat,cosine,10,0.931210,0.135904,0.947661,0.963344
7,bert_finetuned,faiss_flat,cosine,20,0.925764,0.269382,0.964398,0.963686
8,bert_finetuned,faiss_flat,dot,1,0.954800,0.014086,0.954800,0.954800
9,bert_finetuned,faiss_flat,dot,5,0.935859,0.068519,0.946455,0.963000
